In [ ]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

print("Imports done")

Imports done


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Load cleaned stock data from GitHub (download it first to Drive or upload directly)
from google.colab import files
print("Please upload cleaned_stock_data.csv from the GitHub repo")
uploaded = files.upload()

df = pd.read_csv('cleaned_stock_data.csv', header=[0,1], index_col=0)
print("Shape:", df.shape)
print("Columns:", df.columns.tolist()[:6], "...")
print("Data loaded")

Mounted at /content/drive
Please upload cleaned_stock_data.csv from the GitHub repo


Saving cleaned_stock_data.csv to cleaned_stock_data.csv
Shape: (1254, 15)
Columns: [('Close', 'AAPL'), ('Close', 'MSFT'), ('Close', 'SPY'), ('High', 'AAPL'), ('High', 'MSFT'), ('High', 'SPY')] ...
Data loaded


In [ ]:
# Flatten MultiIndex columns
df.columns = ['_'.join(col).strip() for col in df.columns.values]
df.index = pd.to_datetime(df.index)
df.reset_index(inplace=True)
df.rename(columns={'index': 'Date'}, inplace=True)

print("Flattened columns:", df.columns.tolist())

# Separate by ticker
tickers = ['AAPL', 'MSFT', 'SPY']
dfs = {}

for ticker in tickers:
    cols = ['Date'] + [c for c in df.columns if ticker in c]
    ticker_df = df[cols].copy()
    ticker_df.columns = [c.replace(f'_{ticker}', '') for c in ticker_df.columns]
    ticker_df['Ticker'] = ticker
    dfs[ticker] = ticker_df
    print(f"  {ticker}: {ticker_df.shape}")

# Combine all tickers into one master dataframe
df_master = pd.concat(dfs.values(), ignore_index=True)
print(f"\nMaster shape: {df_master.shape}")
print(df_master.head())
print("Done")

Flattened columns: ['Date', 'Close_AAPL', 'Close_MSFT', 'Close_SPY', 'High_AAPL', 'High_MSFT', 'High_SPY', 'Low_AAPL', 'Low_MSFT', 'Low_SPY', 'Open_AAPL', 'Open_MSFT', 'Open_SPY', 'Volume_AAPL', 'Volume_MSFT', 'Volume_SPY']
  AAPL: (1254, 7)
  MSFT: (1254, 7)
  SPY: (1254, 7)

Master shape: (3762, 7)
        Date      Close       High        Low       Open     Volume Ticker
0 2021-06-30  133.50210  133.94073  132.43962  132.73204   63261400   AAPL
1 2021-07-01  133.80417  133.86266  132.33229  133.15110   52485800   AAPL
2 2021-07-02  136.42627  136.46526  134.27206  134.41827   78852600   AAPL
3 2021-07-06  138.43428  139.53574  136.53352  136.53352  108181800   AAPL
4 2021-07-07  140.91988  141.23180  139.05809  139.91586  104911600   AAPL
Done


In [ ]:
# Add technical indicators for each ticker
def add_indicators(df):
    df = df.sort_values('Date').copy()

    # Moving averages
    df['SMA_50']  = df['Close'].rolling(50).mean().round(2)
    df['SMA_200'] = df['Close'].rolling(200).mean().round(2)
    df['EMA_50']  = df['Close'].ewm(span=50).mean().round(2)

    # Daily return
    df['Daily_Return'] = df['Close'].pct_change().round(4)

    # Cumulative return
    df['Cumulative_Return'] = ((1 + df['Daily_Return']).cumprod() - 1).round(4)

    # Bollinger Bands
    df['BB_Mid']   = df['Close'].rolling(20).mean().round(2)
    df['BB_Upper'] = (df['BB_Mid'] + 2 * df['Close'].rolling(20).std()).round(2)
    df['BB_Lower'] = (df['BB_Mid'] - 2 * df['Close'].rolling(20).std()).round(2)

    # Volatility (20-day rolling std of returns)
    df['Volatility_20d'] = df['Daily_Return'].rolling(20).std().round(4)

    return df

# Apply to each ticker
result = []
for ticker in ['AAPL', 'MSFT', 'SPY']:
    t_df = df_master[df_master['Ticker'] == ticker].copy()
    t_df = add_indicators(t_df)
    result.append(t_df)

df_final = pd.concat(result, ignore_index=True)
print("Final shape:", df_final.shape)
print("Columns:", df_final.columns.tolist())
print("Indicators added")

Final shape: (3762, 16)
Columns: ['Date', 'Close', 'High', 'Low', 'Open', 'Volume', 'Ticker', 'SMA_50', 'SMA_200', 'EMA_50', 'Daily_Return', 'Cumulative_Return', 'BB_Mid', 'BB_Upper', 'BB_Lower', 'Volatility_20d']
Indicators added


In [ ]:
# Export 1 — Master historical + indicators (main Power BI table)
df_final.to_csv('stock_historical_for_BI.csv', index=False)
print("✓ Exported: stock_historical_for_BI.csv")

# Export 2 — Daily returns comparison (all 3 tickers side by side)
returns_pivot = df_final.pivot_table(
    index='Date', columns='Ticker', values='Daily_Return'
).reset_index()
returns_pivot.columns = ['Date', 'AAPL_Return', 'MSFT_Return', 'SPY_Return']
returns_pivot.to_csv('stock_returns_for_BI.csv', index=False)
print("✓ Exported: stock_returns_for_BI.csv")

# Export 3 — Monthly summary (for trend charts in Power BI)
df_final['Month'] = pd.to_datetime(df_final['Date']).dt.to_period('M').astype(str)
monthly_summary = df_final.groupby(['Month', 'Ticker']).agg(
    Avg_Close=('Close', 'mean'),
    Avg_Volume=('Volume', 'mean'),
    Monthly_Return=('Daily_Return', 'sum'),
    Avg_Volatility=('Volatility_20d', 'mean')
).round(4).reset_index()
monthly_summary.to_csv('stock_monthly_summary_for_BI.csv', index=False)
print("✓ Exported: stock_monthly_summary_for_BI.csv")

print("\nAll 3 CSVs ready for Power BI!")

✓ Exported: stock_historical_for_BI.csv
✓ Exported: stock_returns_for_BI.csv
✓ Exported: stock_monthly_summary_for_BI.csv

All 3 CSVs ready for Power BI!


In [ ]:
from google.colab import files

csvs = [
    'stock_historical_for_BI.csv',
    'stock_returns_for_BI.csv',
    'stock_monthly_summary_for_BI.csv'
]

for csv in csvs:
    files.download(csv)
    print(f"Downloaded: {csv}")

print("\nAll CSVs downloaded! Ready for Power BI.")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Downloaded: stock_historical_for_BI.csv


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Downloaded: stock_returns_for_BI.csv


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Downloaded: stock_monthly_summary_for_BI.csv

All CSVs downloaded! Ready for Power BI.
